# NumPy: O idioma Global das Redes Neurais

Para a maioria dos casos, o PyTorch aceita objetos de imagem do PIL diretamente. Mas, nem sempre na área de Deep Learning receberemos dados exclusivamente do PIL. Por exemplo, se utilizarmos CV2 para ler imagens em tempo real pela webcam, receberemos os dados no formato de matrizes numéricas multidimensionais do NumPy. Por isso, é muito importante passar por essa etapa.

Antes de continuarmos, é conveniente processar a imagem inicial para um estado mais fácil de trabalhar. Abaixo, lemos a foto, convertemos para modo RGB com tamanho máximo (300, 300) e salvamos em outro caminho.

In [1]:
from PIL import Image
from notebooks.config import DATA_DIR

image = Image.open(DATA_DIR / "folha-cafe.png")
image = image.convert("RGB")
image.thumbnail((300, 300))

image.save(DATA_DIR / "folha-cafe-processada.jpg")

## Tensores

No NumPy, temos o chamado ndarray que é um array n-dimensional que é capaz de representar diversos formatos de organização de dados como vetores, matrizes, tensores 3D e tensores 4D. Um vetor é uma lista simples, um array unidimensional. Uma matriz é uma lista com listas simples dentro, ou seja, um array bidimensional. Quando chegamos no array tridimensional, já podemos chamar de tensor. É uma lista com listas que contém listas de números por exemplo, ou, melhor dizendo, um array de matrizes. Por fim, chegamos no tensor 4D que é como uma matriz de matrizes. Para entender melhor, eis uma aplicação de cada formato.

- Vetor: Lista de compras
- Matriz: Sistema de coordenadas
- Tensor 3D: Foto colorida
- Tensor 4D: Lote de fotos coloridas

Imagine que em uma foto colorida temos a reta da largura e a reta da altura, tendo assim uma matriz. Essa matriz guarda apenas a intensidade da cor vermelha da foto colorida. Mas, uma foto colorida tem dados de intensidade para as cores verde e azul também. Dessa forma, temos mais duas matrizes. Por isso, um tensor 3D representaria uma única foto colorida.

Quando trabalharmos com PyTorch, teremos de utilizar lotes de fotos coloridas que serão enviadas de uma vez para a placa de vídeo ou processador. Esses lotes de fotos coloridas são basicamente vetores de fotos coloridas. Conhecidos como tensores 4D.

### Exemplos de Tensores

In [2]:
array = [1, 2, 3, 4, 5]

matriz = [
    [1, 2],
    [3, 4]
]

tensor3d = [
    [
        [1, 2],
        [3, 4]
    ], 

    # ...
]

tensor4d = [
    [
        [
            [1, 2],
            [3, 4]
        ], 
        [
            [5, 6],
            [7, 8]
        ], 
        [
            [9, 10],
            [11, 12]
        ], 
    ],

    # ...
]

## Imagens como Tensores

Para gerar um tensor a partir de uma imagem, utilizamos o método array da biblioteca numpy, passando como primeiro parâmetro a imagem do PIL. 

O tensor terá algumas propriedades descritivas como dtype que informa o tipo primitivo dos dados armazenados no tensor. 

A propriedade shape descreve o formato do tensor. A primeira camada tem 300 itens, um para cada pixel da altura. A segunda camada tem 225 itens, um para cada pixel da largura. A terceira camada tem 3 itens, um para a intensidade de vermelho, um para a intensidade de verde e outro para a intensidade de azul. 

A propriedade ndim informa o número de dimensões do array, que nesse caso é tridimensional. Também podemos perceber isso porque a tupla que descreve o formato do array tem três itens.

Outra propriedade é a size, que informa a quantidade de valores que de fato estão guardados no array. No exemplo da imagem, são 202.500 valores. Sendo eles a intensidade de vermelho, verde e azul para cada um dos pixels da imagem.

In [3]:
import numpy

tensor = numpy.array(image)

print(f"Tipo primitivo: {tensor.dtype}")
print(f"Formato da matriz: {tensor.shape}")
print(f"Número de dimensões: {tensor.ndim}")
print(f"Total de valores: {tensor.size}")


Tipo primitivo: uint8
Formato da matriz: (300, 225, 3)
Número de dimensões: 3
Total de valores: 202500


Quando começarmos a aplicar convoluções, vários números de 0 até 255 (que representam as intensidades das cores) serão somados. O tipo padrão de imagens no NumPy é uint8, ou seja, números inteiros que podem ser representados em até 8 bits, variando de 0 à 255. Se tentarmos somar por exemplo 250 + 10, o resultado passa de 255. Se isso acontecer e tentarmos representar em 8 bits, o valor final será o número 4, e não 260. Além disso, ao somar vários números calculando uma média, o resultado provavelmente não será inteiro.

O padrão da indústria é converter o tipo de dado para float com 32 bits. Para isso, utilizamos o método astype do tensor, passando como argumento o tipo float32 do numpy. Um outro padrão da indústria é normalizar os valores dividindo por 255. Assim, todas as médias de intensidades de cada cor em uma convolução passarão a residir entre 0 e 1, sendo números com muitas casas decimais. Por terem muitas casas decimais, também precisam ser representados utilizando muitos bits. 

In [4]:
tensor_float = tensor.astype(numpy.float32) / 255

## Corte de Imagens

Caso você esteja se perguntando. Sim, é possível cortar imagens utilizando o numpy, e é incrivelmente simples. O numpy constrói os tensores de uma forma que possibilita fazer isso facilmente. Abaixo, definimos uma nova variável que guardará um pedaço do tensor original. Utilizando a sintaxe do python, podemos selecionar intervalos do tensor. No exemplo, selecionamos na primeira camada os índices de 37 até 262. Lembra que a primeira camada do tensor guarda os dados para cada pixel da altura da imagem? Isso significa que uma linha do tensor contém todos os dados para uma linha completa da imagem. Ao selecionar um subconjunto, estamos efetivamente selecionando um intervalo de pixels. No exemplo abaixo, selecionamos do pixel 38 até o pixel 262, totalizando 225 pixels. 

Mas porque o pixel 37 é incluso e o 262 não é? Porque incluir o primeiro e não incluir o último torna mais fácil entender a quantidade de itens que está sendo selecionada. Se fizermos simplesmente o início do intervalo subtraído do final, temos a quantidade de pixels de resultado. Como a imagem tem dimensões 225x300, esse código efetivamente selecionará exatamente o mesmo quadrado central 225x225 que selecionamos anteriormente com PIL, já que 37 subtraído de 262 é exatamente 225.

In [5]:
from notebooks.config import ROOT_DIR

FIGURES_DIR = ROOT_DIR / "assets" / "images"

tensor_crop = tensor[37:262, :, :]

image_crop = Image.fromarray(tensor_crop)

image_crop.save(FIGURES_DIR / "figura_07.jpg")

![Figura 7](assets/images/figura_07.jpg)

Outro exemplo. Neste, a partir do corte central de 225x225, selecionamos um quadrado 45x45 exatamente no centro dele.

In [6]:
tensor_crop = tensor_crop[90:135, 90:135, :]

image_crop = Image.fromarray(tensor_crop)

image_crop.save(FIGURES_DIR / "figura_08.jpg")

![Figura 8](assets/images/figura_08.jpg)

## Aplicando Máscaras

Podemos aplicar máscaras de cores nos tensores. No exemplo abaixo, definimos as variáveis R, G e B, cara uma guardando todos os valores de vermelho, verde e azul do tensor. No python temos os operadores condicionais "and" e "or" que fazem comparações lógicas escalares. Ou seja, um valor comparado à outro valor. Há também os operadores "&" (and) e "|" (or) que são comparadores vetorizados. Abaixo, criamos uma condição que verifica se os tons de verde são maiores que os tons de vermelho, maiores que os tons de azul e se são maiores que 0.2. Para unir todas as condições, utilizamos o operador "&", já que estamos comparando vetores. Esta condição é a máscara. A máscara fará uma comparação pixel por pixel de toda a imagem. A condição só será verdadeira para um pixel se todas as condições forem verdadeiras. Ou seja: se o pixel representar um tom de verde.

Utilizando o método count_nonzero do numpy, conseguimos contar a quantidade de pixels que passam pelo filtro da máscara. Quando aplicamos uma máscara condicional, o resultado é sempre True ou False. Mas, como os tensores do numpy idealmente devem sempre guardar apenas números, True vira 1 e False vira 0. Por isso utilizamos count_nonzero. Depois, podemos calcular a quantidade total de pixels da imagem usando os valores da propriedade shape do tensor e calcular a porcentagem de pixels verdes da imagem.

Na prática, é difícil vermos uma aplicação das máscaras como essa. Imagine que em um algoritmo de identificação de objetos em uma imagem tenhamos a resposta do modelo e o gabarito. O gabarito será uma caixa e o algoritmo de identificação contornará o objeto identificado em uma caixa. Podemos utilizar uma máscara para calcular por exemplo que porcentagem do gabarito a resposta do modelo conseguiu cobrir.

In [7]:
R = tensor_float[:, :, 0]
G = tensor_float[:, :, 1]
B = tensor_float[:, :, 2]

green_mask = (G > R) & (G > B) & (G > 0.2)
green_pixel_count = numpy.count_nonzero(green_mask)
image_pixel_count = tensor_float.shape[0] * tensor_float.shape[1]
green_percent = green_pixel_count / image_pixel_count

print(f"{green_percent:.2%}")

88.09%


Outra funcionalidade que podemos aproveitar é a inversão de máscaras. No exemplo abaixo, fizemos uma cópia do tensor para criar um outro tensor. Nesse tensor, utilizamos colchetes com a máscara definida para selecionar os pixels que batem na condição. Mas, usamos o operador "~" que inverte a máscara. Então na prática, essa linha de código seleciona todos os pixels que não passam na condição da máscara. Por fim, atribuímos o valor zero para todos esses pixels. O que acontecerá é que no tensor, todos os pixels que não apresentam um tom predominantemente verdes ficarão pretos. 

In [8]:
tensor_segmented = tensor_float.copy()
tensor_segmented[~green_mask] = 0.0
tensor_segmented = tensor_segmented * 255
tensor_segmented = tensor_segmented.astype(numpy.uint8)

image_segmented = Image.fromarray(tensor_segmented)
image_segmented.save(FIGURES_DIR / "figura_09.jpg")

![Figura 9](assets/images/figura_09.jpg)

Perceba que as lesões que a folha de café tinha ficaram todas pretas. Em ambiente controlado, essa máscara poderia ser útil para identificar a área afetada. Mas, a folha teria de estar isolada em uma bancada com fundo branco, por exemplo. Dessa forma, poderíamos identificar a área da foto com tons de marrom.

## NumPy para PyTorch

Vamos relembrar como o numpy monta um tensor que representa uma imagem colorida. Perceba que a tupla de resultado para o formato do tensor retorna o tamanho das dimensões na ordem (H, W, C) em que H é o número de pixels da altura, W é o número de pixels da largura e C é o número de canais.

In [9]:
print(tensor_float.shape)

(300, 225, 3)


O PyTorch exige uma ordem diferente (C, H, W) porque as GPUs realizam convoluções lendo os pixels em blocos contíguos de memória. Então ter todos os valores do canal vermelho juntos, seguindos pelos do canal verde e do canal azul acelera brutalmente o processo de cálculo do produto escalar dos filtros convolucionais.

Agora, pare um momento e pense na dificuldade que seria ter de reordenar o tensor para que os canais estivessem na primeira camada, a altura estivesse na segunda e a largura estivesse na terceira. Parece verdadeiramente um bicho de sete cabeças, não parece? Porque de uma certa forma, é exatamente um bixo de sete cabeças. Nossa sorte é que o numpy oferece uma solução muito fácil para isso. 

Chamos essa operação de transposição. Para realizar a transposição, utilizamos o método transpose do numpy, que recebe como primeiro parâmetro o tensor a partir do qual um novo tensor reordenado será gerado. Como segundo argumento, o método recebe uma tupla que será a responsável por definir a nova ordem das dimensões do tensor. O primeiro item da tupla define qual camada passará a ser a primeira. O segundo item define qual camada passará a ser a segunda e o terceiro item define qual camada passará a ser a terceira. No exemplo abaixo, movemos a terceira camada para a primeira, a primeira para a segunda e a segunda para a terceira. Dessa forma, o que antes era (H, W, C) agora é (C, H, W). Tome um momento para visualizar o que aconteceu com o tensor. Agora, ao invés de a primeira camada ter 300 itens, ela tem apenas 3 itens. Cada item guarda todos os valores para um único canal de cores.

In [10]:
tensor_transposed = numpy.transpose(tensor_float, (2, 0, 1))

print(tensor_transposed.shape)

(3, 300, 225)


### Contiguidade

Quando um tensor não transposto é guardado na memória, ele é guardado de forma contígua. Imagine a memória como uma linha reta de gavetas. Para ler um array contíguo, basta ler as gavetas de forma sequencial. Quando o numpy realiza a transposição de um tensor, a permutação dos dados nas gavetas da memória continua exatamente a mesma, mas a forma de leitura muda. Então por exemplo quando pedirmos a primeira linha da matriz transposta, o primeiro número pode estar na primeira gaveta e o segundo pode estar na quarta gaveta. Isso é chamado de memória não contígua. As GPUs não gostam de ficar saltando gavetas. A GPU gosta de engolir blocos inteiros de gavetas vizinhas de uma vez só ao aplicar convoluções. Se tentarmos entregar um tensor armazenado em memória não contígua, o PyTorch reclamará logo de cara.

A solução padrão para esse problema é passar o tensor armazenado em memória não contígua para o método ascontiguousarray do numpy que modifica a disposição dos blocos para que voltem a estar em formato contíguo.

In [11]:
tensor_contiguous = numpy.ascontiguousarray(tensor_transposed)

### Dimensão de Lote

Modelos de deep learning nunca aceitam tensores 3d que representam uma única imagem. Eles operam sempre com base em tensores 4d que são lotes de imagens no formato (B, C, H, W) onde B é o tamanho do lote. Para adicionar a dimensão B, podemos utilizar o método expand_dims do numpy, que recebe como primeiro parâmetro o tensor que terá o número de dimensões expandido. O parâmetro aixs define em qual índice da tupla de formato do tensor a nova dimensão de tamanho 1 será inserida. No Exemplo abaixo, inserimos no índice 0, já que a dimensão batch tem de ser a primeira. 

Também podemos fazer de outra forma

Observe agora o formato do tensor 4d exatamente na ordem (B, C, H, W).



In [12]:
image_batch = numpy.expand_dims(tensor_contiguous, axis=0)
# OU
# image_batch = tensor_contiguous[numpy.newaxis, ...]

print(image_batch.shape)

(1, 3, 300, 225)


Agora, vamos supor que você tem um tensor 4d de uma única imagem, como no exemplo anterior, e quer convertê-lo para um tensor 3d de volta para representar a imagem solta. Para isso, podemos utilizar o método squeeze do numpy. Passando como primeiro parâmetro o tensor a ser reduzido e como parâmetro axis a dimensão do tensor que queremos remover. No exemplo abaixo, removemos a primeira dimensão do tensor image_batch, que era exatamente o tamanho do lote. Dessa forma, image_tensor3d volta a ser exatamente aquela imagem única do início.

In [13]:
image_tensor3d = numpy.squeeze(image_batch, axis=0)

print(image_tensor3d.shape)

(3, 300, 225)


### Ponte entre NumPy e PyTorch

Para evitar gargalos de processamento, devemos transferir os bytes entre NumPy e PyTorch sem copiá-los na memória RAM. Para isso, podemos utilizar o método from_numpy do PyTorch como no exemplo abaixo. O método receberá o tensor contíguo e retornará um tensor atrelado ao processador. O tensor retornado apontará para os exatos mesmos endereços na memória RAM que o tensor contíguo do NumPy. Portanto, modificar o tensor contíguo também modificaria o novo tensor do PyTorch.

In [14]:
import torch

tensor_pytorch = torch.from_numpy(tensor_contiguous)

Quando um tensor passa pelas camadas de uma rede neural, o PyTorch grava tudo que aconteceu com os pixels em cada camada para poder utilizar esse histórico posteriormente para realizar a backpropagation. Por isso, quando queremos converter um tensor do PyTorch de volta para o NumPy após passar pela rede neural, temos de chamar o método detach do tensor do PyTorch, que cortará as "cordas" com o gravador, jogando fora o histórico que agora não é mais necessário. 

Quando passamos dados através de uma rede neural, geralmente fazemos isso utilizando a placa de vídeo (CUDA). Para continuar a conversão de volta para NumPy, precisamos trazer o tensor de volta da placa de vídeo para o processador. Para isso, chamamos o método cpu encadeado ao detach.

Por fim, chamamos o método numpy encadeado ao detach e cpu para converter o tensor do PyTorch de volta para o formato de um tensor do NumPy. 

In [15]:
tensor_numpy = tensor_pytorch.detach().cpu().numpy()